### Unit standardization for OSI, PIP, and PEEP (if Vent_Type is CDGR, then multiply by 1.01972)

In [3]:
import pandas as pd

# === Load the original Excel sheet ===
file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_df.xlsx"
df = pd.read_excel(file_path, sheet_name="Sheet9")

# === Identify OSI, PIP, and PEEP columns ===
columns_to_scale = [col for col in df.columns if any(key in col for key in ["OSI", "PIP", "PEEP"])]

# === Create a copy of the DataFrame to modify ===
df_scaled = df.copy()

# === Apply scaling factor to CDGR ventilator rows only ===
scale_factor = 1.01972
cdgr_mask = df_scaled["Ventilation Type"] == "CDGR"

df_scaled.loc[cdgr_mask, columns_to_scale] = (
    df_scaled.loc[cdgr_mask, columns_to_scale] * scale_factor
)

# === Save to Sheet10 ===
with pd.ExcelWriter(file_path, mode="a", if_sheet_exists="replace") as writer:
    df_scaled.to_excel(writer, sheet_name="Sheet10", index=False)

print("✅ Scaling complete. Modified data saved to Sheet10.")


✅ Scaling complete. Modified data saved to Sheet10.


### 1D-CNN (PIP, PEEP, TV, and Flow)

In [ ]:
# import os
# import re
# import pandas as pd
# import numpy as np
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from glob import glob

# # === Set CUDA device ===
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print("🔧 Using device:", device)

# # === CNN Model ===
# class Simple1DCNN(nn.Module):
#     def __init__(self):
#         super(Simple1DCNN, self).__init__()
#         self.conv1 = nn.Conv1d(1, 16, kernel_size=5, padding=2)
#         self.bn1 = nn.BatchNorm1d(16)
#         self.pool1 = nn.MaxPool1d(2)

#         self.conv2 = nn.Conv1d(16, 32, kernel_size=5, padding=2)
#         self.bn2 = nn.BatchNorm1d(32)
#         self.pool2 = nn.MaxPool1d(2)

#         self.conv3 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
#         self.bn3 = nn.BatchNorm1d(64)
#         self.pool3 = nn.AdaptiveAvgPool1d(1)

#     def forward(self, x):
#         x = self.pool1(F.relu(self.bn1(self.conv1(x))))
#         x = self.pool2(F.relu(self.bn2(self.conv2(x))))
#         x = self.pool3(F.relu(self.bn3(self.conv3(x))))
#         return x.view(x.size(0), -1)

# # === Setup ===
# root_dir = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_1st_Raw/Study23_Tag282_EventList/RENAMED/PEEP_Settings_V3_1st/Abnormal_Deletion_V3_1st/TWs_V3_1st/TV_V3_1st"
# cnn = Simple1DCNN().to(device).eval()

# # Columns needing conversion for CDGR scaling
# conversion_signals = {
#     "CDGR - PIP", "CDGR - PEEP",
#     "CAPSULE_DRAGERMEDIBUS_VITAL_2564", "CAPSULE_DRAGERMEDIBUS_VITAL_1189"
# }
# conversion_factor = 1.01972

# # Candidate columns to search in each file
# vital_signals = {
#     "pip": ["AVEA - PIP", "CDGR - PIP", "SVU - PIP",
#             "CAPSULE_AVEAA_VITAL_2564", "CAPSULE_DRAGERMEDIBUS_VITAL_2564", "CAPSULE_MAQUETC_VITAL_1414"],
#     "peep": ["AVEA - PEEP", "CDGR - PEEP", "SVU - PEEP",
#              "CAPSULE_AVEAA_VITAL_1189", "CAPSULE_DRAGERMEDIBUS_VITAL_1189", "CAPSULE_MAQUETC_VITAL_1189"],
#     "tv": ["TV (mL/Kg)"]
# }
# wave_signals = {
#     "flow": ["CDGR - Flow", "AVEA - Air Flow Wave", "SVU - Flow",
#              "CAPSULE_AVEAA_WAVE_52911", "CAPSULE_DRAGERMEDIBUS_WAVE_118", "MAQUET_CAPSULE_AIRWAY_FLOW_WAVE"]
# }

# def extract_signal(df, candidates):
#     """Extract first available candidate column as a 1D float array, drop NaNs, apply conversion if needed."""
#     if df is None or df.empty:
#         return None

#     df_clean = df.dropna(how="all")  # drop fully-NaN rows
#     for c in candidates:
#         if c in df_clean.columns and df_clean[c].notna().any():
#             sig = pd.to_numeric(df_clean[c], errors="coerce").dropna().to_numpy(dtype=float)
#             if sig.size == 0:
#                 continue
#             if c in conversion_signals:
#                 sig = sig * conversion_factor
#             return sig
#     return None

# def ensure_tw_numeric(df):
#     """Normalize Tumbling_window to numeric (handles 'TW1', '1', 1.0)."""
#     if "Tumbling_window" not in df.columns:
#         return df
#     tw_raw = df["Tumbling_window"]
#     if tw_raw.dtype == object:
#         tw_num = tw_raw.astype(str).str.extract(r"(\d+)", expand=False)
#         df["Tumbling_window"] = pd.to_numeric(tw_num, errors="coerce")
#     else:
#         df["Tumbling_window"] = pd.to_numeric(df["Tumbling_window"], errors="coerce")
#     return df

# # === Gather files (single type) ===
# all_files = sorted(glob(os.path.join(root_dir, "*.csv")))
# print(f"📄 Found {len(all_files)} CSV files under: {root_dir}")

# # === Process each file ===
# all_features = []

# for path in all_files:
#     base = os.path.basename(path).replace(".csv", "")

#     try:
#         df = pd.read_csv(path)
#         df = ensure_tw_numeric(df)

#         if "Tumbling_window" not in df.columns:
#             print(f"⚠️ Missing Tumbling_window: {base}")
#             continue

#         for tw in range(1, 7):
#             df_tw = df[df["Tumbling_window"] == tw]
#             features = []

#             # PIP, PEEP, TV from same df_tw
#             for key in ["pip", "peep", "tv"]:
#                 sig = extract_signal(df_tw, vital_signals[key])
#                 if sig is not None and len(sig) >= 64:
#                     x = torch.tensor(sig, dtype=torch.float32).view(1, 1, -1).to(device)
#                     with torch.no_grad():
#                         feat = cnn(x).cpu().numpy().flatten()
#                     features.append(feat)
#                 else:
#                     features.append(np.zeros(64, dtype=float))

#             # Flow from same df_tw
#             flow_sig = extract_signal(df_tw, wave_signals["flow"])
#             if flow_sig is not None and len(flow_sig) >= 64:
#                 x = torch.tensor(flow_sig, dtype=torch.float32).view(1, 1, -1).to(device)
#                 with torch.no_grad():
#                     feat = cnn(x).cpu().numpy().flatten()
#                 features.append(feat)
#             else:
#                 features.append(np.zeros(64, dtype=float))

#             row_id = f"{base}_TW{tw}"
#             all_feat = np.concatenate(features)  # 4 signals × 64 = 256
#             all_features.append([row_id] + all_feat.tolist())
#             print(f"✅ Processed: {row_id}")

#     except Exception as e:
#         print(f"❌ Error processing {path}: {e}")

# # === Save to Excel ===
# columns = ["RowID"] + [f"f{i+1}" for i in range(256)]
# df_cnn = pd.DataFrame(all_features, columns=columns)

# output_path = os.path.join(root_dir, "CNN_Features_All.xlsx")
# df_cnn.to_excel(output_path, index=False)
# print("✅ CNN features saved to:", output_path)


🔧 Using device: cuda
📄 Found 188 CSV files under: /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_1st_Raw/Study23_Tag282_EventList/RENAMED/PEEP_Settings_V3_1st/Abnormal_Deletion_V3_1st/TWs_V3_1st/TV_V3_1st
✅ Processed: 1000780_20250420_18_1st_Raw_TW1
✅ Processed: 1000780_20250420_18_1st_Raw_TW2
✅ Processed: 1000780_20250420_18_1st_Raw_TW3
✅ Processed: 1000780_20250420_18_1st_Raw_TW4
✅ Processed: 1000780_20250420_18_1st_Raw_TW5
✅ Processed: 1000780_20250420_18_1st_Raw_TW6
✅ Processed: 1014659_20241012_15_1st_Raw_TW1
✅ Processed: 1014659_20241012_15_1st_Raw_TW2
✅ Processed: 1014659_20241012_15_1st_Raw_TW3
✅ Processed: 1014659_20241012_15_1st_Raw_TW4
✅ Processed: 1014659_20241012_15_1st_Raw_TW5
✅ Processed: 1014659_20241012_15_1st_Raw_TW6
✅ Processed: 1020005_20241229_22_1st_Raw_TW1
✅ Processed: 1020005_20241229_22_1st_Raw_TW2
✅ Processed: 1020005_20241229_22_1st_Raw_TW3
✅ Processed: 1020005_20241229_22_1st_Raw_TW4
✅ Processed: 1020005_20241229_22_1st_Raw_TW5
✅ Processed

02/17/2026

============================================================

V3 CNN FEATURE EXTRACTION (FIXED)
- Loads the SAME CNN checkpoint used for V1/V2 feature extraction
- Ensures deterministic inference (eval mode)
- Saves features to Excel

============================================================

In [1]:
import os
import re
import hashlib
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from glob import glob

# === Set CUDA device ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("🔧 Using device:", device)

# === CNN Model ===
class Simple1DCNN(nn.Module):
    def __init__(self):
        super(Simple1DCNN, self).__init__()
        self.conv1 = nn.Conv1d(1, 16, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(16)
        self.pool1 = nn.MaxPool1d(2)

        self.conv2 = nn.Conv1d(16, 32, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(32)
        self.pool2 = nn.MaxPool1d(2)

        self.conv3 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(64)
        self.pool3 = nn.AdaptiveAvgPool1d(1)

    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))
        return x.view(x.size(0), -1)

# === Setup ===
root_dir = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_1st_Raw/Study23_Tag282_EventList/RENAMED/PEEP_Settings_V3_1st/Abnormal_Deletion_V3_1st/TWs_V3_1st/TV_V3_1st"
os.makedirs(root_dir, exist_ok=True)

# === IMPORTANT: Load SAME checkpoint as V1/V2 ===
# Update this path to the checkpoint you saved in the V1/V2 extraction script
cnn_ckpt = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/V1V2_1st/Simple1DCNN_fixed_seed0_20260217.pt"

if not os.path.exists(cnn_ckpt):
    raise FileNotFoundError(
        f"❌ CNN checkpoint not found:\n  {cnn_ckpt}\n"
        "Make sure you generated/saved it from the V1/V2 script first."
    )

cnn = Simple1DCNN().to(device)
cnn.load_state_dict(torch.load(cnn_ckpt, map_location=device))
cnn.eval()
print("✅ Loaded V1/V2 CNN checkpoint:", cnn_ckpt)

# Optional: print md5 fingerprint so you can verify this matches V1/V2
def md5_of_file(path, chunk_size=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

print("✅ CNN checkpoint MD5:", md5_of_file(cnn_ckpt))

# Columns needing conversion for CDGR scaling
conversion_signals = {
    "CDGR - PIP", "CDGR - PEEP",
    "CAPSULE_DRAGERMEDIBUS_VITAL_2564", "CAPSULE_DRAGERMEDIBUS_VITAL_1189"
}
conversion_factor = 1.01972

# Candidate columns to search in each file
vital_signals = {
    "pip": ["AVEA - PIP", "CDGR - PIP", "SVU - PIP",
            "CAPSULE_AVEAA_VITAL_2564", "CAPSULE_DRAGERMEDIBUS_VITAL_2564", "CAPSULE_MAQUETC_VITAL_1414"],
    "peep": ["AVEA - PEEP", "CDGR - PEEP", "SVU - PEEP",
             "CAPSULE_AVEAA_VITAL_1189", "CAPSULE_DRAGERMEDIBUS_VITAL_1189", "CAPSULE_MAQUETC_VITAL_1189"],
    "tv": ["TV (mL/Kg)"]
}
wave_signals = {
    "flow": ["CDGR - Flow", "AVEA - Air Flow Wave", "SVU - Flow",
             "CAPSULE_AVEAA_WAVE_52911", "CAPSULE_DRAGERMEDIBUS_WAVE_118", "MAQUET_CAPSULE_AIRWAY_FLOW_WAVE"]
}

def extract_signal(df, candidates):
    """Extract first available candidate column as a 1D float array, drop NaNs, apply conversion if needed."""
    if df is None or df.empty:
        return None

    df_clean = df.dropna(how="all")  # drop fully-NaN rows
    for c in candidates:
        if c in df_clean.columns and df_clean[c].notna().any():
            sig = pd.to_numeric(df_clean[c], errors="coerce").dropna().to_numpy(dtype=float)
            if sig.size == 0:
                continue
            if c in conversion_signals:
                sig = sig * conversion_factor
            return sig
    return None

def ensure_tw_numeric(df):
    """Normalize Tumbling_window to numeric (handles 'TW1', '1', 1.0)."""
    if "Tumbling_window" not in df.columns:
        return df
    tw_raw = df["Tumbling_window"]
    if tw_raw.dtype == object:
        tw_num = tw_raw.astype(str).str.extract(r"(\d+)", expand=False)
        df["Tumbling_window"] = pd.to_numeric(tw_num, errors="coerce")
    else:
        df["Tumbling_window"] = pd.to_numeric(df["Tumbling_window"], errors="coerce")
    return df

# === Gather files (single type) ===
all_files = sorted(glob(os.path.join(root_dir, "*.csv")))
print(f"📄 Found {len(all_files)} CSV files under: {root_dir}")

# === Process each file ===
all_features = []

for path in all_files:
    base = os.path.basename(path).replace(".csv", "")

    try:
        df = pd.read_csv(path)
        df = ensure_tw_numeric(df)

        if "Tumbling_window" not in df.columns:
            print(f"⚠️ Missing Tumbling_window: {base}")
            continue

        for tw in range(1, 7):
            df_tw = df[df["Tumbling_window"] == tw]
            features = []

            # PIP, PEEP, TV from same df_tw
            for key in ["pip", "peep", "tv"]:
                sig = extract_signal(df_tw, vital_signals[key])
                if sig is not None and len(sig) >= 64:
                    x = torch.tensor(sig, dtype=torch.float32).view(1, 1, -1).to(device)
                    with torch.no_grad():
                        feat = cnn(x).cpu().numpy().flatten()
                    features.append(feat)
                else:
                    features.append(np.zeros(64, dtype=float))

            # Flow from same df_tw
            flow_sig = extract_signal(df_tw, wave_signals["flow"])
            if flow_sig is not None and len(flow_sig) >= 64:
                x = torch.tensor(flow_sig, dtype=torch.float32).view(1, 1, -1).to(device)
                with torch.no_grad():
                    feat = cnn(x).cpu().numpy().flatten()
                features.append(feat)
            else:
                features.append(np.zeros(64, dtype=float))

            row_id = f"{base}_TW{tw}"
            all_feat = np.concatenate(features)  # 4 signals × 64 = 256
            all_features.append([row_id] + all_feat.tolist())
            print(f"✅ Processed: {row_id}")

    except Exception as e:
        print(f"❌ Error processing {path}: {e}")

# === Save to Excel ===
columns = ["RowID"] + [f"f{i+1}" for i in range(256)]
df_cnn = pd.DataFrame(all_features, columns=columns)

output_path = os.path.join(root_dir, "CNN_Features_All_20260217.xlsx")
df_cnn.to_excel(output_path, index=False)
print("✅ CNN features saved to:", output_path)


🔧 Using device: cpu
✅ Loaded V1/V2 CNN checkpoint: /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/V1V2_1st/Simple1DCNN_fixed_seed0_20260217.pt
✅ CNN checkpoint MD5: e488ca2bd62da22be2550ddfc3ad3ee3
📄 Found 188 CSV files under: /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_1st_Raw/Study23_Tag282_EventList/RENAMED/PEEP_Settings_V3_1st/Abnormal_Deletion_V3_1st/TWs_V3_1st/TV_V3_1st
✅ Processed: 1000780_20250420_18_1st_Raw_TW1
✅ Processed: 1000780_20250420_18_1st_Raw_TW2
✅ Processed: 1000780_20250420_18_1st_Raw_TW3
✅ Processed: 1000780_20250420_18_1st_Raw_TW4
✅ Processed: 1000780_20250420_18_1st_Raw_TW5
✅ Processed: 1000780_20250420_18_1st_Raw_TW6
✅ Processed: 1014659_20241012_15_1st_Raw_TW1
✅ Processed: 1014659_20241012_15_1st_Raw_TW2
✅ Processed: 1014659_20241012_15_1st_Raw_TW3
✅ Processed: 1014659_20241012_15_1st_Raw_TW4
✅ Processed: 1014659_20241012_15_1st_Raw_TW5
✅ Processed: 1014659_20241012_15_1st_Raw_TW6
✅ Processed: 1020005_2024122

Pivot to wide format

In [5]:
import pandas as pd

# === Load the CNN feature file ===
# cnn_feature_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_1st_Raw/Study23_Tag282_EventList/RENAMED/PEEP_Settings_V3_1st/Abnormal_Deletion_V3_1st/TWs_V3_1st/TV_V3_1st/CNN_Features_All.xlsx"
cnn_feature_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_1st_Raw/Study23_Tag282_EventList/RENAMED/PEEP_Settings_V3_1st/Abnormal_Deletion_V3_1st/TWs_V3_1st/TV_V3_1st/CNN_Features_All_20260217.xlsx"

df_cnn = pd.read_excel(cnn_feature_path, sheet_name="Sheet1")

# === Extract base RowID (without _TWx) ===
df_cnn["BaseID"] = df_cnn["RowID"].str.replace(r"_TW\d+$", "", regex=True)
df_cnn["Window"] = df_cnn["RowID"].str.extract(r"_TW(\d)$")[0].astype(int)

# === Sort by BaseID and Tumbling Window ===
df_cnn = df_cnn.sort_values(by=["BaseID", "Window"])

# === Create wide-format dataframe ===
merged_rows = []
for base_id, group in df_cnn.groupby("BaseID"):
    group = group.sort_values("Window")
    
    # Only merge if all 6 windows are present
    if group.shape[0] != 6:
        print(f"⚠️ Skipping {base_id} — only {group.shape[0]} windows found.")
        continue

    feature_arrays = []
    for window in range(1, 7):
        row = group[group["Window"] == window]
        if row.empty:
            feature_arrays.extend([np.nan] * 256)
            # feature_arrays.extend([np.nan] * 64)
        else:
            feature_arrays.extend(row.iloc[0, 1:257].values)
            # feature_arrays.extend(row.iloc[0, 1:65].values)

    merged_rows.append([base_id] + feature_arrays)

# === Generate full column names: f1_TW1 to f256_TW6 ===
new_columns = ["RowID"]
for tw in range(1, 7):
    new_columns.extend([f"f{i}_TW{tw}" for i in range(1, 257)])
    # new_columns.extend([f"f{i}_TW{tw}" for i in range(1, 65)])

# === Build final dataframe ===
final_df = pd.DataFrame(merged_rows, columns=new_columns)

# === Save to Excel ===
output_path = cnn_feature_path.replace(".xlsx", "_MergedByPatient_20260217.xlsx")
final_df.to_excel(output_path, index=False)
print("✅ Merged CNN features saved to:", output_path)


✅ Merged CNN features saved to: /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_1st_Raw/Study23_Tag282_EventList/RENAMED/PEEP_Settings_V3_1st/Abnormal_Deletion_V3_1st/TWs_V3_1st/TV_V3_1st/CNN_Features_All_20260217_MergedByPatient_20260217.xlsx


Merge the CNN features to V3_df

In [6]:
import re
import pandas as pd
import numpy as np

# === File paths ===
xlsx_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_df.xlsx"
cnn_feature_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_1st_Raw/Study23_Tag282_EventList/RENAMED/PEEP_Settings_V3_1st/Abnormal_Deletion_V3_1st/TWs_V3_1st/TV_V3_1st/CNN_Features_All_20260217_MergedByPatient_20260217.xlsx"

# === Load Excel sheets ===
df_v3 = pd.read_excel(xlsx_path, sheet_name="Sheet12")
df_cnn = pd.read_excel(cnn_feature_path, sheet_name="Sheet1")

# --- Checks ---
if "1st_rel_fname" not in df_v3.columns:
    raise ValueError("Sheet12 is missing required column: '1st_rel_fname'")
if "RowID" not in df_cnn.columns:
    raise ValueError("CNN feature sheet is missing required column: 'RowID'")

# ------------------------------------------------------------
# Create MatchKey (normalize BOTH sides)
# df_v3 1st_rel_fname looks like: 223_20250711_19_1st
# df_cnn RowID looks like: 1000780_20250420_18_1st_Raw_TW3 OR 1000780_20250420_18_1st_Raw
# Goal MatchKey: 1000780_20250420_18_1st
# ------------------------------------------------------------
df_v3 = df_v3.copy()
df_cnn = df_cnn.copy()

df_v3["MatchKey"] = df_v3["1st_rel_fname"].astype(str).str.strip()

df_cnn["MatchKey"] = (
    df_cnn["RowID"].astype(str).str.strip()
    .str.replace(r"_TW\d+$", "", regex=True)   # remove trailing _TW#
    .str.replace("_Raw", "", regex=False)      # remove _Raw
)

# ------------------------------------------------------------
# Identify CNN feature columns like f1_TW1 ... f256_TW6
# (We will SUBSTITUTE these in Sheet12 when writing Sheet14)
# ------------------------------------------------------------
feature_pattern = re.compile(r"^f\d+_TW\d+$")
cnn_feature_cols = [c for c in df_cnn.columns if feature_pattern.match(c)]

if len(cnn_feature_cols) == 0:
    raise ValueError("No CNN feature columns found like f*_TW* in CNN sheet.")

print(f"✅ Found {len(cnn_feature_cols)} CNN feature columns in df_cnn")

# ------------------------------------------------------------
# Deduplicate CNN by MatchKey:
# Keep first row that has ANY non-NaN across CNN feature columns
# ------------------------------------------------------------
def pick_best_row(group: pd.DataFrame) -> pd.Series:
    non_all_nan = group[cnn_feature_cols].notna().any(axis=1)
    if non_all_nan.any():
        # take first non-all-nan row in original order
        return group.loc[non_all_nan.idxmax()]
    return group.iloc[0]

df_cnn_best = (
    df_cnn.groupby("MatchKey", as_index=False)
          .apply(lambda g: pick_best_row(g), include_groups=False)
)

# Index for fast lookup (features only)
df_cnn_clean = df_cnn_best.set_index("MatchKey")[cnn_feature_cols]

# ------------------------------------------------------------
# Build CNN feature matrix aligned to df_v3 order
# ------------------------------------------------------------
default_feat = {c: np.nan for c in cnn_feature_cols}

cnn_features = []
for key in df_v3["MatchKey"]:
    if key in df_cnn_clean.index:
        cnn_features.append(df_cnn_clean.loc[key].to_dict())
    else:
        cnn_features.append(default_feat.copy())

df_feat = pd.DataFrame(cnn_features, index=df_v3.index)

# ------------------------------------------------------------
# SUBSTITUTE old CNN columns in Sheet12-copy:
# If Sheet12 already contains f*_TW* columns, DROP them first, then add new ones.
# Sheet12 itself is NOT overwritten; we write the substituted copy to Sheet14.
# ------------------------------------------------------------
existing_cols = [c for c in cnn_feature_cols if c in df_v3.columns]
if existing_cols:
    print(f"🔁 Replacing {len(existing_cols)} existing CNN feature columns in Sheet12-copy")
    df_v3_no_old = df_v3.drop(columns=existing_cols)
else:
    df_v3_no_old = df_v3

df_sheet14 = pd.concat([df_v3_no_old, df_feat], axis=1)

# Optional match stats
matched = int(df_v3["MatchKey"].isin(df_cnn_clean.index).sum())
print(f"✅ Matched rows: {matched} / {len(df_v3)}")

# ------------------------------------------------------------
# Save as Sheet14 (Sheet12 untouched)
# ------------------------------------------------------------
with pd.ExcelWriter(xlsx_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    df_sheet14.to_excel(writer, sheet_name="Sheet14", index=False)

print("✅ Sheet14 created from Sheet12 with CNN features substituted (f*_TW*).")


✅ Found 1536 CNN feature columns in df_cnn
🔁 Replacing 1536 existing CNN feature columns in Sheet12-copy
✅ Matched rows: 148 / 148
✅ Sheet14 created from Sheet12 with CNN features substituted (f*_TW*).


### Add Severity_Label

In [16]:
import pandas as pd

# === Load Excel ===
excel_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_df.xlsx"
df = pd.read_excel(excel_path, sheet_name="Sheet11")

# === Define required columns ===
target_col = "OSI_V3_12th_avg"

base_template = [
    "OSI_mean_TW{}", "OSI_std_TW{}", "PIP_mean_TW{}", "PIP_std_TW{}",
    "PEEP_mean_TW{}", "PEEP_std_TW{}", "TV_mean_TW{}(mL/Kg)", "TV_std_TW{}(mL/Kg)",
    "Avg_NegFlowDur_TW{}", "Std_NegFlowDur_TW{}", "Avg_PeakInterval_TW{}", "Std_PeakInterval_TW{}"
]

tw_features = {
    i: [f.format(i) for f in base_template] + [f"w{i}_SubBandEnergy_row{j}" for j in range(1, 17)]
    for i in range(1, 7)
}

# === CNN feature columns to require as well ===
cnn_columns = []
for tw in range(1, 7):
    cnn_columns.extend([f"f{i}_TW{tw}" for i in range(1, 257)])

required_columns = [target_col] + [
    col for features in tw_features.values() for col in features
] + cnn_columns

# --- Check for missing columns (helps catch naming mismatches early) ---
missing_cols = [c for c in required_columns if c not in df.columns]
if missing_cols:
    raise ValueError(
        f"Sheet11 is missing {len(missing_cols)} required columns. "
        f"First 25 missing: {missing_cols[:25]}"
    )

# === Drop rows with missing values in required columns ===
filtered_df = df.dropna(subset=required_columns).copy()

# === Assign Severity_Label based on OSI_V3_12th_avg ===
def classify_severity(val):
    if val < 7.5:
        return "mild"
    elif val < 12.3:
        return "moderate"
    else:
        return "severe"

filtered_df["Severity_Label"] = filtered_df[target_col].apply(classify_severity)

# === Save to Sheet12 ===
with pd.ExcelWriter(excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    filtered_df.to_excel(writer, sheet_name="Sheet12", index=False)

print("✅ Filtered data with Severity_Label saved to Sheet12.")
print("Rows kept after filtering:", filtered_df.shape[0])


✅ Filtered data with Severity_Label saved to Sheet12.
Rows kept after filtering: 148


Subset the validation set according to 1st_Time_Start (After 02-28-2025)

In [8]:
from openpyxl import load_workbook
import pandas as pd

file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_df.xlsx"

df = pd.read_excel(file_path, sheet_name="Sheet14")

df_subset = df[df["1st_Time_Start"] > "2025-02-28"].copy()

# === Save to Sheet15 ===
with pd.ExcelWriter(file_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    df_subset.to_excel(writer, sheet_name="Sheet15", index=False)

print("Rows kept after filtering:", df_subset.shape)

Rows kept after filtering: (65, 1844)
